# 篮球录像分析 · 源码版

带授权的原片和同源观测文件可以运行本笔记本。此源码版不附带赛事画面或观测数据。首次练习可运行同目录的 movelab-measurement.ipynb。

依赖 Python 3.11+、OpenCV、NumPy、IPython 与 FFmpeg。把有权限使用的对应原片和证据放进 basketball 目录。

In [ ]:
from pathlib import Path
from bisect import bisect_right
import hashlib, json, subprocess
import cv2
import numpy as np
from IPython.display import display, Image, Video

folders = [Path('basketball'), Path('lab/basketball')]
assets = next(p for p in folders if (p/'match-evidence.json').is_file())
def load(name):
    source = assets/f'{name}-source.mp4'
    data = json.loads((assets/f'{name}-evidence.json').read_text(encoding='utf-8'))
    with source.open('rb') as stream:
        if hashlib.file_digest(stream, 'sha256').hexdigest()!=data['sourceSha256']:
            raise ValueError('原片指纹不匹配。')
    return source,data

LIME,CYAN,ORANGE,INK=(80,250,215),(255,220,90),(80,170,255),(20,30,23)
def point(p):return tuple(int(round(v)) for v in p)
def line(out,path,color,w=1):
    if len(path)>1:
        cv2.polylines(out,[np.array([point(p) for p in path],np.int32)],False,color,w,cv2.LINE_AA)
def label(out,value,x,y,color=(235,241,232),scale=.45):
    x,y=point((x,max(16,y)))
    (w,h),_=cv2.getTextSize(value,cv2.FONT_HERSHEY_SIMPLEX,scale,1)
    cv2.rectangle(out,(x,y-h-3),(x+w+6,y+3),INK,-1)
    cv2.putText(out,value,(x+3,y),cv2.FONT_HERSHEY_SIMPLEX,scale,color,1,cv2.LINE_AA)
def dot(out,p,color,r=4):
    cv2.circle(out,point(p),r,INK,-1,cv2.LINE_AA)
    cv2.circle(out,point(p),r,color,1,cv2.LINE_AA)
def sample_at(rows,idx,frame,age):
    i=bisect_right(idx,frame)-1
    return rows[i] if i>=0 and frame-idx[i]<=age else None

def render(name,overlay,size,hop,poster_index):
    source,data=load(name)
    cap=cv2.VideoCapture(str(source))
    if not cap.isOpened():raise ValueError('无法读取原片。')
    fps=cap.get(cv2.CAP_PROP_FPS)
    poster,video=[assets/f'{name}-review.{ext}' for ext in ('jpg','mp4')]
    ffmpeg=['ffmpeg','-y','-loglevel','error','-f','rawvideo','-pix_fmt','bgr24',
            '-s',f'{size[0]}x{size[1]}','-r',str(fps/hop),'-i','pipe:0',
            '-an','-c:v','libx264','-pix_fmt','yuv420p','-crf','24',
            '-movflags','+faststart',str(video)]
    index,count=0,0
    try:
        with subprocess.Popen(ffmpeg,stdin=subprocess.PIPE) as encoder:
            try:
                while True:
                    ok,frame=cap.read()
                    if not ok:break
                    if index%hop==0:
                        shown=overlay(frame,index)
                        encoder.stdin.write(shown.tobytes())
                        if index==poster_index and not cv2.imwrite(str(poster),shown):
                            raise OSError('无法保存证据截图。')
                        count+=1
                    index+=1
            finally:encoder.stdin.close()
            if encoder.wait()!=0 or not count:raise RuntimeError('生成视频失败。')
    finally:cap.release()
    print(f'{count} frames rendered')
    display(Image(filename=str(poster)))
    display(Video(filename=str(video),embed=True))

## 01 球员检测、跟踪、号码与球场站位

对照 7.60 秒。蓝框是临时检测轨迹，橙框是同帧号码 OCR 裁剪，亮黄线是模型场线，俯视图的点由场地锚点和单应矩阵得到。检测轨迹 ID 与场地锚点来自不同观测层。号码带问号表示单帧读数，这个片段没有形成连续三次一致的号码证据。

[Roboflow 原版 Colab · RF-DETR、SAM2、球队聚类、号码与站位](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/basketball-ai-how-to-detect-track-and-identify-basketball-players.ipynb)



In [ ]:
_,match=load('match')
rows=match['frames']
indices=[r['frameIndex'] for r in rows]
court_by_frame={r['frame']:r for r in match['court']}

def project(matrix,p):
    x,y,w=np.asarray(matrix)@np.array([*p,1.])
    return None if abs(w)<1e-9 else (x/w,y/w)

def match_overlay(frame,index):
    out=np.full((450,1064,3),INK,np.uint8)
    out[:,:800]=cv2.resize(frame,(800,450),interpolation=cv2.INTER_AREA)
    raw=lambda p:tuple(v*.625 for v in p)
    row=sample_at(rows,indices,index,4)
    court=court_by_frame.get(index)
    label(out,'01 / PLAYER REVIEW',814,31,LIME,.62)
    label(out,f'SOURCE {index/match["sourceFps"]:.2f}s | F{index}',814,55)
    if court:
        for path in court['referenceLines']:line(out,[raw(p) for p in path],LIME)
        sx,sy=match['courtSize']
        mini=lambda p:(824+p[0]/sx*216,108+p[1]/sy*114)
        for path in match['courtPaths']:line(out,[mini(p) for p in path],(121,152,126))
        H=court.get('imageToCourt')
        for anchor in court['positions']:
            p=project(H,anchor['anchorPixels']) if H else None
            if p and 0<=p[0]<=sx and 0<=p[1]<=sy:
                dot(out,mini(p),LIME)
                dot(out,raw(anchor['anchorPixels']),LIME)
        label(out,'COURT / ground anchors',814,90,LIME)
    else:
        label(out,'COURT / mapping unavailable',814,90,ORANGE)
        cv2.rectangle(out,(824,108),(1040,222),(60,75,63),1)
        label(out,'No positions drawn',847,168)
    if row:
        label(out,f'TRACK SAMPLE {row["tMs"]/1000:.2f}s',814,254,CYAN)
        for obj in row['objects']:
            if obj['label']!='person':continue
            a,b,c,d=obj['bbox']
            cv2.rectangle(out,point(raw((a,b))),point(raw((c,d))),CYAN,1)
            label(out,f'ID {obj["trackId"]}',*raw((a,b)),CYAN,.40)
        # Keep OCR crops on their exact evidence frame.
        if row['frameIndex']==index:
            for i,ocr in enumerate([r for r in row['ocr'] if r.get('number')][:5]):
                a,b,c,d=ocr['cropPixels']
                cv2.rectangle(out,point(raw((a,b))),point(raw((c,d))),ORANGE,2)
                label(out,f'{ocr["number"]}?',*raw((c,d)),ORANGE)
                label(out,f'ID {ocr["trackId"]}  OCR {ocr["number"]}?',814,280+i*18,ORANGE)
    label(out,'Court lines and anchors',814,392,LIME)
    label(out,'Temporary track',814,413,CYAN)
    label(out,'Single-frame jersey reading',814,434,ORANGE)
    return out

print(len(court_by_frame),'accepted court frames')
render('match',match_overlay,(1064,450),1,190)

## 02 投篮姿势、球轨迹与篮筐

浅绿线连接可见关节，蓝框为人工复核过的篮筐区域，橙线跟随已关联的球。右腕与球分离给出 1.201 秒的离手候选；肘、膝角用原画面像素算二维夹角，低可见度时留空。篮筐附近的两条轨迹在 2.102–2.603 秒之间无法可靠接续，结果保留未知。

[vision-demos 投篮原版 · 姿势、球和篮筐](https://github.com/jeremyipark/vision-demos/tree/main/basketball)



In [ ]:
_,shot=load('shot')
shot_rows=shot['frames']
shot_indices=[r['frameIndex'] for r in shot_rows]
release=shot['release'][0]
edges=[(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]

def angle(pose,ids):
    ps=[pose[i] for i in ids]
    if min(p[2] for p in ps)<.55:return '--'
    a,b,c=(np.array(p[:2])*[shot['width'],shot['height']] for p in ps)
    u,v=a-b,c-b
    norm=np.linalg.norm(u)*np.linalg.norm(v)
    if norm<1e-8:return '--'
    return f'{np.degrees(np.arccos(np.clip(u@v/norm,-1,1))):.0f} deg'

def shot_overlay(frame,index):
    out=np.full((608,720,3),INK,np.uint8)
    out[:,:342]=cv2.resize(frame,(342,608),interpolation=cv2.INTER_AREA)
    screen=lambda p:(p[0]*342,p[1]*608)
    ms=index*1000/shot['sourceFps']
    label(out,'02 / SHOT REVIEW',362,35,LIME,.70)
    label(out,f'SOURCE {ms/1000:.3f}s | FRAME {index}',362,65)
    row=sample_at(shot_rows,shot_indices,index,6)
    if ms<=shot['rimUntilMs']:
        rim=[screen((p['x'],p['y'])) for p in shot['rim']]
        line(out,rim+[rim[0]],CYAN,2)
        label(out,'RIM ROI',*rim[0],CYAN)
    if row:
        person=next((p for p in row['objects'] if p['trackId']==release['personTrackId'] and p['label']=='person'),{})
        pose=person.get('keypoints',[])
        if len(pose)==17:
            for a,b in edges:
                if min(pose[a][2],pose[b][2])>=.55:line(out,[screen(pose[a]),screen(pose[b])],LIME,2)
            for p in pose:
                if p[2]>=.55:dot(out,screen(p),LIME,3)
            label(out,'Right elbow: '+angle(pose,(6,8,10)),362,118,LIME,.58)
            label(out,'Right knee: '+angle(pose,(12,14,16)),362,151,LIME,.58)
        label(out,f'POSE SAMPLE {row["tMs"]/1000:.3f}s / IMAGE PLANE',362,183)
    # Only connect within each associated fragment. The gap stays open.
    for track in shot['tracks']:
        if track['track'] not in {'B2','B4'}:continue
        pts=[screen((p['x'],p['y'])) for p in track['points'] if p['tMs']<=ms]
        line(out,pts,ORANGE,2)
        for p in pts:dot(out,p,ORANGE)
    early=[r for r in release['evidence'] if r['tMs']<=ms]
    line(out,[screen((r['x'],r['y'])) for r in early],ORANGE,2)
    for r in early:dot(out,screen((r['x'],r['y'])),ORANGE)
    if early:
        r=early[0]
        line(out,[screen((r['wristX'],r['wristY'])),screen((r['x'],r['y']))],CYAN,2)
    gap=shot['gaps'][0]
    label(out,f'RELEASE CANDIDATE {release["tMs"]/1000:.3f}s',362,242,ORANGE,.58)
    label(out,f'NEAR RIM {gap["beforeMs"]/1000:.3f}s - {gap["afterMs"]/1000:.3f}s',362,282,CYAN,.55)
    cv2.rectangle(out,(362,308),(695,390),(39,59,43),-1)
    label(out,'RESULT / UNKNOWN',378,340,ORANGE,.65)
    label(out,'Gap in the linked ball observations',378,372)
    label(out,'Visible joints',362,455,LIME,.55)
    label(out,'Observed ball path',362,485,ORANGE,.55)
    label(out,'Reviewed rim region',362,515,CYAN,.55)
    return out

print(len(shot['release']),'release candidate; result',shot['gaps'][0]['outcome'])
render('shot',shot_overlay,(720,608),2,126)

### 自己复核

- 暂停在比赛的 7.60 秒，从检测框、号码裁剪、俯视锚点各找一个可核对证据。再选一个机位映射失效的帧，检查地图为何清空。
- 投篮段在 2.102–2.603 秒出现断轨。把同帧截图与原片放在一起，讨论能否确认进球，以及需要哪一帧的新证据。